# SLD on parcae — GPU (validated adapter)

**Speculative Looped Decoding (SLD)** accelerates a looped transformer by
verifying *future loop states in parallel* instead of walking the recurrence one
step at a time. On CPU ([SLD repo](https://github.com/asystemoffields/SLD),
`RESULTS.md`) SLD is *exactly lossless* and collapses a depth-`k` synthetic
recurrence to **one** sequential core round (vs `k`). The CPU wall-clock win is
fragile (a batched verify of many states leaves the CPU "flat" region); **a GPU
fixes that** — the batched verify is one parallel kernel.

This notebook runs on the real [`parcae-140m`](https://github.com/sandyresearch/parcae)
stable looped LM (recurrence `T=8`, contractive core that converges). The
`ParcaeLoop` adapter below is the **same code already validated on CPU** in
`bench/parcae_sld.py` — it asserts the reconstructed loop matches parcae's native
output before any acceleration claim. We then measure, on GPU:

1. **convergence** — how few loops parcae actually needs (its redundancy);
2. **lossless early-exit vs SLD** — sequential core rounds;
3. **wall-clock at batch 1 / 16 / 64** — where parallel depth-verification beats
   the sequential walk (the regime CPU cannot show).

On parcae's *short* `T=8` loop, sequential early-exit already captures much of the
CPU headroom; SLD's distinct win is **parallelism at batch** and **deep**
recurrence (e.g. Huginn's 32–132 unrolls). The synthetic repo results are where
the lossless depth-collapse is shown cleanly; parcae confirms a real stable looped
LM carries large exploitable recurrent redundancy.

In [ ]:
import torch, time, statistics as st
assert torch.cuda.is_available(), "Runtime ▸ Change runtime type ▸ GPU"
DEV = "cuda"; torch.manual_seed(0)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# parcae's REAL package is the GitHub source (the PyPI 'parcae-lm' is an empty stub).
!pip -q install "git+https://github.com/sandyresearch/parcae" einops safetensors tokenizers transformers >/dev/null 2>&1
import parcae_lm
m = parcae_lm.from_pretrained("SandyResearch/parcae-140m").to(DEV).eval()
for p in m.parameters(): p.requires_grad_(False)
T = int(getattr(m.config, "mean_recurrence", 8))
print("loaded parcae-140m | recurrence T =", T,
      "| layers prelude/core/coda =",
      m.config.n_layers_in_prelude, m.config.n_layers_in_recurrent_block, m.config.n_layers_in_coda)

## 1. The validated parcae loop adapter

`encode` (embeds + prelude + `ln_prelude` + `initialize_state`), `step`
(`core_block_forward`, which applies parcae's diagonal input-injection and the
shared core; time-invariant given `_current_input_ids`), `decode`
(`C → coda → ln_f → lm_head·logit_scale`). The cell **asserts** the manual loop
reproduces parcae's native `forward(num_steps_pair=[T,0])` output.

In [ ]:
class ParcaeLoop:
    def __init__(self, m):
        self.m = m
        self.off_coda = m.config.n_layers_in_prelude + m.config.n_layers_in_recurrent_block
        self.logit_scale = m.config.init.logit_scale
    @torch.no_grad()
    def encode(self, ids):
        m = self.m; m._current_input_ids = ids
        fc = m.freqs_cis[:, : ids.shape[1]]
        e = m.transformer.wte(ids)
        if getattr(m, "emb_scale", 1) != 1: e = e * m.emb_scale
        for i, blk in enumerate(m.transformer.prelude):
            ve = m.value_embeds[str(i)](ids) if str(i) in m.value_embeds else None
            e = blk(e, fc, None, ve=ve)
        if m.config.prelude_norm: e = m.transformer.ln_prelude(e)
        return m.initialize_state(e), e, fc
    @torch.no_grad()
    def step(self, x, e, fc):
        return self.m.core_block_forward(x, e, fc, None, torch.tensor(0, device=x.device), torch.tensor(0, device=x.device))
    @torch.no_grad()
    def decode(self, x, ids, fc):
        m = self.m; x = m.transformer.C(x)
        for i, blk in enumerate(m.transformer.coda):
            k = str(self.off_coda + i)
            ve = m.value_embeds[k](ids) if k in m.value_embeds else None
            x = blk(x, fc, None, ve=ve)
        x = m.transformer.ln_f(x)
        return (m.lm_head(x).float() * self.logit_scale)[:, -1, :]

loop = ParcaeLoop(m)

# self-validation: manual loop == native forward
g = torch.Generator(device=DEV).manual_seed(0)
ok = True
for _ in range(4):
    ids = torch.randint(0, 30000, (1, 12), generator=g, device=DEV)
    x, e, fc = loop.encode(ids)
    for _ in range(T): x = loop.step(x, e, fc)
    ours = loop.decode(x, ids, fc).argmax(-1)
    native = m(ids, num_steps_pair=torch.tensor([T, 0]), return_logits=True)["logits"][:, -1, :].argmax(-1)
    ok = ok and bool((ours == native).all())
assert ok, "adapter does not reproduce native parcae output — check CORE/injection assumptions"
print("[ok] adapter validated on real parcae (GPU).")

## 2. parcae converges — quantify the redundancy

In [ ]:
@torch.no_grad()
def settle_loops(ids):
    toks = [m(ids, num_steps_pair=torch.tensor([t, 0]), return_logits=True)["logits"][:, -1, :].argmax(-1)
            for t in range(1, T + 1)]
    final = toks[-1]; s = T
    for t in range(T, 0, -1):
        if (toks[t-1] == final).all(): s = t
        else: break
    return s

g = torch.Generator(device=DEV).manual_seed(1)
ss = [settle_loops(torch.randint(0, 30000, (1, 16), generator=g, device=DEV)) for _ in range(16)]
print(f"mean loops parcae actually needs: {st.mean(ss):.2f} / {T}  -> ~{T - st.mean(ss):.1f} redundant")

## 3. Lossless early-exit vs SLD (sequential core rounds)

In [ ]:
@torch.no_grad()
def aitken(a, b, c):
    d1, d2 = b - a, c - b; dd = d2 - d1
    coef = (d2*dd).flatten(1).sum(1, keepdim=True) / (dd*dd).flatten(1).sum(1, keepdim=True).clamp_min(1e-9)
    return c - coef.view(-1, *([1]*(c.dim()-1))) * d2

@torch.no_grad()
def early_exit(ids, patience=2):
    x, e, fc = loop.encode(ids); prev = None; stable = 0
    for t in range(1, T + 1):
        x = loop.step(x, e, fc); cur = loop.decode(x, ids, fc).argmax(-1)
        if prev is not None and (cur == prev).all():
            stable += 1
            if stable >= patience: return cur, t
        else: stable = 0
        prev = cur
    return prev, T

@torch.no_grad()
def sld(ids, warmup=3, verify=2):
    x, e, fc = loop.encode(ids); hs = [x]; r = 0
    for _ in range(warmup): x = loop.step(x, e, fc); hs.append(x); r += 1
    while r < T:
        s = aitken(hs[-3], hs[-2], hs[-1]) if len(hs) >= 3 else hs[-1]
        tok = loop.decode(s, ids, fc).argmax(-1); cur = s; good = True
        for _ in range(verify):
            cur = loop.step(cur, e, fc); r += 1
            if (loop.decode(cur, ids, fc).argmax(-1) != tok).any(): good = False; break
        if good: return tok, r
        hs.append(cur)
    return loop.decode(hs[-1], ids, fc).argmax(-1), r

g = torch.Generator(device=DEV).manual_seed(2)
ids_set = [torch.randint(0, 30000, (1, 16), generator=g, device=DEV) for _ in range(16)]
ee_r, ee_l, sl_r, sl_l = [], 0, [], 0
for ids in ids_set:
    full = m(ids, num_steps_pair=torch.tensor([T,0]), return_logits=True)["logits"][:,-1,:].argmax(-1)
    a, r = early_exit(ids); ee_r.append(r); ee_l += int((a==full).all())
    a, r = sld(ids);        sl_r.append(r); sl_l += int((a==full).all())
print(f"full-loop: {T} rounds")
print(f"early-exit: mean {st.mean(ee_r):.2f} rounds, lossless {ee_l}/{len(ids_set)}")
print(f"SLD:        mean {st.mean(sl_r):.2f} rounds, lossless {sl_l}/{len(ids_set)}")

## 4. GPU wall-clock at batch — the parallel-verify win

Time the **recurrence** at batch 1 / 16 / 64. The key SLD primitive is verifying
several depths in ONE batched core call; on GPU that batched call is parallel, so
fewer *sequential* rounds is a real latency drop that **holds as batch grows** —
unlike CPU, where the verify batch leaves the flat region. We compare the full
`T`-loop recurrence against a 1-round batched verify-`K` core pass.

In [ ]:
@torch.no_grad()
def time_ms(fn, B, reps=20, warm=5):
    g = torch.Generator(device=DEV).manual_seed(3)
    ids = torch.randint(0, 30000, (B, 32), generator=g, device=DEV)
    for _ in range(warm): fn(ids); torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(reps): fn(ids); torch.cuda.synchronize()
    return (time.perf_counter() - t) / reps * 1e3

@torch.no_grad()
def full_recurrence(ids):
    x, e, fc = loop.encode(ids)
    for _ in range(T): x = loop.step(x, e, fc)
    return loop.decode(x, ids, fc)

@torch.no_grad()
def sld_recurrence(ids, warmup=3, K=4):
    # warmup real steps, then ONE batched core call verifying K candidate depths
    x, e, fc = loop.encode(ids); hs = [x]
    for _ in range(warmup): x = loop.step(x, e, fc); hs.append(x)
    s = aitken(hs[-3], hs[-2], hs[-1])
    cand = [s] + hs[-(K - 1):]                              # K candidate states
    batch = torch.stack(cand, 0)                            # [K,B,T,d]
    Kn = batch.shape[0]
    flat = batch.reshape(Kn * batch.shape[1], *batch.shape[2:])
    e_rep = e.repeat(Kn, 1, 1)
    saved = loop.m._current_input_ids
    loop.m._current_input_ids = ids.repeat(Kn, 1)          # value-embeds must match the K*B batch
    _ = loop.step(flat, e_rep, fc)                          # one batched (parallel) verify pass
    loop.m._current_input_ids = saved
    return loop.decode(s, ids, fc)

print(f"{'batch':>6} {'full ms':>9} {'SLD ms':>9} {'speedup':>9}")
for B in (1, 16, 64):
    tf = time_ms(lambda x: full_recurrence(x), B)
    ts = time_ms(lambda x: sld_recurrence(x), B)
    print(f"{B:>6} {tf:>9.2f} {ts:>9.2f} {tf/ts:>8.2f}x")

## 5. Reading it honestly

- **Convergence / redundancy** — parcae's loop settles well before `T=8`, so a
  stable looped LM genuinely carries skippable recurrent depth.
- **Lossless early-exit vs SLD rounds** — on this short loop, sequential
  early-exit is strong; SLD matches it in *sequential rounds* and its advantage is
  doing the verification **in parallel** (one batched core pass).
- **Wall-clock at batch** — the SLD pass replaces `T` sequential core calls with
  `warmup + 1` calls (the last a batched parallel verify); on GPU the batched call
  is ~one kernel, so the saving holds as batch grows. This is the thing the CPU
  experiment could not show.

**Where SLD pays off most:** deeper recurrence. parcae uses `T=8`; recurrent-depth
LMs like Huginn unroll 32–132 times. The same `ParcaeLoop`/`sld` apply with larger
`T`, where collapsing the sequential depth is a far bigger latency win. The clean,
*exactly lossless* depth-collapse is in the synthetic SLD repo
(https://github.com/asystemoffields/SLD); this notebook validates the premise and
the machinery on a real model.